# Automatic CCTV calibration and speed workflow

Check this camera view for usable saved calibration. If missing or invalid, infer vehicle geometry
and fit a local road mapping; otherwise reuse the mapping. Then estimate speeds and save outputs.
Fresh video intervals still need tracking and vehicle ground positions. Inference outputs are reused
only when the source hash, interval, configuration and artifact hashes match.

This notebook demonstrates a cold-cache run with fresh GPU inference, then a warm-cache run
that must skip calibration fitting. Calibration files are under `workflow_outputs/calibration_cache`.
Download that directory or supply persistent storage to retain calibration across Kaggle sessions.

**Experimental km/h only. Approximate car dimensions determine scale. Sensitivity ranges are not
confidence intervals; physical speed accuracy remains unvalidated.** Five sampled view checks
cannot guarantee that a camera stayed fixed between samples. This implementation targets the
A3 CH14 camera's two-carriageway geometry; other cameras need their road-region assumptions reviewed.

References: [Sochor et al.](https://arxiv.org/abs/1702.06451),
[UrbanOmniDetect](https://github.com/mkturkcan/urbanomnidetect).


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'ultralytics==8.4.175', 'lap==0.5.13'])


In [ ]:
sources = {'workflow.py': '"""Check saved camera calibration, calibrate when needed, then estimate experimental speeds."""\nimport argparse\nfrom datetime import datetime, timezone\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport uuid\n\nimport calibrate_speed as speed\nfrom camera_cache import (CalibrationCache, file_sha256, canonical_hash, read_views,\n                          stable_view, valid_calibration)\n\nROOT = Path(__file__).resolve().parent\nINFERENCE_VERSION = \'yolo11s-1280-bytetrack-urban1920-v1\'\nGEOMETRY_SHA = \'9aa55280a5ef1c9d89c0acb5baeb974b90fbdf3c1deead1413b24d7af249226d\'\nINPUT_FILES = (\'tracking/tracks.csv\', \'tracking/run_report.json\',\n               \'geometry/observations.json\', \'geometry/geometry_run_report.json\')\n\n\ndef input_identity(video_sha, metadata):\n    return {\'source_sha256\': video_sha,\n            **{key: metadata[key] for key in [\'width\', \'height\', \'fps\', \'start_frame\', \'end_frame\']},\n            \'inference_version\': INFERENCE_VERSION, \'geometry_model_sha256\': GEOMETRY_SHA}\n\n\ndef write_input_manifest(directory, identity):\n    directory = Path(directory)\n    manifest = {\'schema_version\': 1, \'identity\': identity,\n                \'files\': {name: file_sha256(directory / name) for name in INPUT_FILES}}\n    (directory / \'input_manifest.json\').write_text(json.dumps(manifest, indent=2), encoding=\'utf-8\')\n\n\ndef verified_inputs(directory, identity):\n    try:\n        directory = Path(directory)\n        manifest = json.loads((directory / \'input_manifest.json\').read_text(encoding=\'utf-8\'))\n        if manifest[\'schema_version\'] != 1 or manifest[\'identity\'] != identity:\n            return False\n        if not all(file_sha256(directory / name) == manifest[\'files\'][name] for name in INPUT_FILES):\n            return False\n        tracking = json.loads((directory / \'tracking/run_report.json\').read_text())\n        geometry = json.loads((directory / \'geometry/geometry_run_report.json\').read_text())\n        return (tracking[\'source_width\'] == identity[\'width\'] and tracking[\'source_height\'] == identity[\'height\']\n                and tracking[\'source_fps\'] == identity[\'fps\']\n                and tracking[\'processed_frames\'] == identity[\'end_frame\'] - identity[\'start_frame\']\n                and geometry[\'model_sha256\'] == GEOMETRY_SHA\n                and geometry[\'ground_order_vote_agreement\'] >= .8\n                and geometry[\'ground_indices\'] in [list(range(4)), list(range(4,8))]\n                and geometry[\'width\'] == identity[\'width\'] and geometry[\'height\'] == identity[\'height\'])\n    except (OSError, ValueError, KeyError, TypeError):\n        return False\n\n\ndef copy_inputs(source, destination):\n    for name in (*INPUT_FILES, \'input_manifest.json\'):\n        target = destination / name\n        target.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copy2(source / name, target)\n\n\ndef inference_inputs(args, run_dir, identity, work_root, stages):\n    destination = run_dir / \'inputs\'\n    cache = work_root / \'inference_cache\' / canonical_hash(identity)\n    candidates = [cache]\n    if args.saved_inputs:\n        candidates.append(Path(args.saved_inputs))\n    else:\n        candidates.append(ROOT / \'output\' / \'inputs\')\n    for candidate in candidates:\n        if verified_inputs(candidate, identity):\n            copy_inputs(candidate, destination)\n            if candidate != cache:\n                copy_inputs(candidate, cache)\n            stages.extend([{\'stage\': \'tracking\', \'action\': \'reuse_verified_inputs\'},\n                           {\'stage\': \'vehicle_geometry\', \'action\': \'reuse_verified_inputs\'}])\n            print(\'Reusing tracking and vehicle geometry verified for this exact video and interval.\', flush=True)\n            return destination\n    try:\n        for package in [\'ultralytics\', \'torch\', \'pandas\', \'ipython\', \'lap\']:\n            importlib.metadata.version(package)\n    except importlib.metadata.PackageNotFoundError as exc:\n        raise RuntimeError(\'Fresh inference needs requirements-inference.txt or the GPU workflow notebook.\') from exc\n    if importlib.metadata.version(\'ultralytics\') != \'8.4.175\':\n        raise RuntimeError(\'Fresh inference requires ultralytics==8.4.175 for the recorded model configuration.\')\n    environment = os.environ.copy()\n    environment.update({\'FYP_SPEED_SOURCE\': str(Path(args.video).resolve()),\n                        \'FYP_START_S\': str(args.start), \'FYP_DURATION_S\': str(args.duration),\n                        \'FYP_TRACKING_OUTPUT\': str(destination / \'tracking\'),\n                        \'FYP_GEOMETRY_OUTPUT\': str(destination / \'geometry\'),\n                        \'FYP_TRACKS_CSV\': str(destination / \'tracking/tracks.csv\'),\n                        \'FYP_MODELS_DIR\': str(work_root / \'models\'),\n                        \'FYP_SKIP_INSTALL\': \'1\', \'MPLBACKEND\': \'Agg\'})\n    if args.device:\n        environment[\'FYP_DEVICE\'] = args.device\n    for stage, script in [(\'tracking\', \'notebook/baseline.py\'),\n                          (\'vehicle_geometry\', \'geometry_notebook/extract_geometry.py\')]:\n        print(\'Running \' + stage + \' inference...\', flush=True)\n        subprocess.run([sys.executable, \'-X\', \'utf8\', str(ROOT / script)],\n                       env=environment, cwd=ROOT, check=True)\n        stages.append({\'stage\': stage, \'action\': \'model_inference\'})\n    write_input_manifest(destination, identity)\n    if not verified_inputs(destination, identity):\n        raise RuntimeError(\'Inference output metadata did not match the requested video and interval.\')\n    copy_inputs(destination, cache)\n    return destination\n\n\ndef run_workflow(args):\n    started = time.monotonic()\n    video = Path(args.video).resolve()\n    if not video.is_file():\n        raise ValueError(\'Source video does not exist.\')\n    if args.start < 0 or args.duration < 2:\n        raise ValueError(\'Use a nonnegative start and a duration of at least two seconds.\')\n    work_root = Path(args.work_dir).resolve()\n    run_id = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%SZ\') + \'-\' + uuid.uuid4().hex[:8]\n    run_dir = work_root / \'runs\' / run_id\n    run_dir.mkdir(parents=True)\n    stages = []\n    report = {\'schema_version\': 1, \'run_id\': run_id, \'camera_id\': args.camera_id,\n              \'source_video\': video.name, \'start_s\': args.start, \'duration_s\': args.duration,\n              \'accuracy_validated\': False, \'stages\': stages, \'status\': \'running\'}\n    code = 0\n    try:\n        metadata, views = read_views(video, args.start, args.duration)\n        report[\'video_metadata\'] = metadata\n        report[\'view_stability\'] = stable_view(views)\n        if not report[\'view_stability\'][\'valid\']:\n            raise ValueError(\'The camera view changed or could not be verified within this interval. \'\n                             \'Use a stable-camera segment; no speeds were generated.\')\n        signature = {\'width\': metadata[\'width\'], \'height\': metadata[\'height\'],\n                     \'geometry_model_sha256\': GEOMETRY_SHA,\n                     \'calculation_code_sha256\': file_sha256(ROOT / \'calibrate_speed.py\'),\n                     \'view_check_code_sha256\': file_sha256(ROOT / \'camera_cache.py\'),\n                     \'nominal_dimensions_m\': [speed.LENGTH_M, speed.WIDTH_M],\n                     \'dimension_scenarios_m\': [list(pair) for pair in speed.DIMENSION_SCENARIOS]}\n        cache = CalibrationCache(args.cache_dir or work_root / \'calibration_cache\', args.camera_id)\n        if args.recalibrate:\n            calibration, decision = None, {\'status\': \'miss\', \'reason\': \'recalibration_requested\'}\n        else:\n            calibration, decision = cache.load(signature, views)\n        report[\'calibration_lookup\'] = decision\n        stages.append({\'stage\': \'calibration_lookup\', \'action\': decision[\'status\'], \'reason\': decision[\'reason\']})\n        print(\'Calibration check: \' + decision[\'reason\'], flush=True)\n        source_sha = file_sha256(video)\n        report[\'source_sha256\'] = source_sha\n        identity = input_identity(source_sha, metadata)\n        inputs = inference_inputs(args, run_dir, identity, work_root, stages)\n        output = run_dir / \'speed\'\n        if calibration is None:\n            print(\'No usable saved calibration. Fitting local road geometry...\', flush=True)\n            geometry_report = json.loads((inputs / \'geometry/geometry_run_report.json\').read_text())\n            observations = json.loads((inputs / \'geometry/observations.json\').read_text())\n            footprints = speed.prepare(observations, geometry_report[\'ground_indices\'],\n                                       metadata[\'width\'], metadata[\'height\'])\n            calibration = speed.fit_road_mappings(footprints, geometry_report)\n            stages.append({\'stage\': \'road_calibration\', \'action\': \'fit_new_mapping\'})\n            if valid_calibration(calibration):\n                cache.save(signature, views[0], calibration,\n                           {\'source_sha256\': source_sha, \'start_s\': args.start, \'duration_s\': args.duration,\n                            \'created_at_utc\': datetime.now(timezone.utc).isoformat()})\n            else:\n                # Still write empty result files, making unavailable speeds explicit.\n                speed.run(calibration=calibration, geometry_dir=inputs / \'geometry\', output_dir=output)\n                report[\'status\'] = \'insufficient_calibration_geometry\'\n                raise ValueError(\'Insufficient consistent vehicle geometry for a road mapping; no speeds available.\')\n        else:\n            stages.append({\'stage\': \'road_calibration\', \'action\': \'reuse_saved_mapping\'})\n            print(\'Reusing the saved road mapping; no calibration fitting is run.\', flush=True)\n        print(\'Calculating speeds inside the calibrated road regions...\', flush=True)\n        _, rows, _ = speed.run(calibration=calibration, geometry_dir=inputs / \'geometry\', output_dir=output,\n                               calibration_reused=decision[\'status\'] == \'hit\')\n        report[\'qualified_track_ids\'] = sorted(row[\'track_id\'] for row in rows if row[\'quality_passed\'])\n        report[\'candidate_track_spans\'] = len(rows)\n        report[\'calibration_reused\'] = decision[\'status\'] == \'hit\'\n        stages.append({\'stage\': \'speed_estimation\', \'action\': \'calculate_from_ground_motion\'})\n        if not args.no_render:\n            environment = os.environ.copy()\n            environment.update({\'FYP_SPEED_SOURCE\': str(video), \'FYP_START_S\': str(args.start),\n                                \'FYP_DURATION_S\': str(args.duration), \'FYP_SPEED_OUTPUT\': str(output),\n                                \'FYP_BASELINE_TRACKS\': str(inputs / \'tracking/tracks.csv\')})\n            subprocess.run([sys.executable, \'-X\', \'utf8\', str(ROOT / \'render_speed.py\')],\n                           env=environment, check=True)\n            stages.append({\'stage\': \'render\', \'action\': \'write_video_and_charts\'})\n        report[\'status\'] = \'complete\' if report[\'qualified_track_ids\'] else \'complete_no_qualified_speeds\'\n    except Exception as exc:\n        if report[\'status\'] == \'running\':\n            report[\'status\'] = \'failed\'\n        report[\'error\'] = str(exc)\n        code = 2\n        print(\'Workflow stopped: \' + str(exc), file=sys.stderr, flush=True)\n    finally:\n        report[\'runtime_s\'] = time.monotonic() - started\n        (run_dir / \'workflow_report.json\').write_text(json.dumps(report, indent=2), encoding=\'utf-8\')\n        print(\'Run folder: \' + str(run_dir), flush=True)\n        print(\'Workflow status: \' + report[\'status\'], flush=True)\n    return code, run_dir\n\n\ndef parser():\n    result = argparse.ArgumentParser(description=__doc__)\n    result.add_argument(\'--video\', required=True, help=\'Original video path\')\n    result.add_argument(\'--camera-id\', required=True, help=\'Stable identifier for this CCTV camera\')\n    result.add_argument(\'--start\', type=float, default=150, help=\'Source interval start, seconds\')\n    result.add_argument(\'--duration\', type=float, default=60, help=\'Interval duration, seconds\')\n    result.add_argument(\'--work-dir\', default=str(ROOT / \'outputs\'), help=\'Run, inference and cache storage\')\n    result.add_argument(\'--cache-dir\', help=\'Optional persistent camera-calibration storage\')\n    result.add_argument(\'--saved-inputs\', help=\'Optional verified inference bundle, with input_manifest.json\')\n    result.add_argument(\'--device\', help=\'Inference device, e.g. 0 or cpu\')\n    result.add_argument(\'--recalibrate\', action=\'store_true\', help=\'Refit even if a valid calibration exists\')\n    result.add_argument(\'--no-render\', action=\'store_true\', help=\'Write speed tables/reports without regenerating video\')\n    return result\n\n\nif __name__ == \'__main__\':\n    code, _ = run_workflow(parser().parse_args())\n    raise SystemExit(code)\n', 'camera_cache.py': '"""Conservative camera-view checks and a versioned, integrity-checked calibration cache."""\nimport hashlib\nimport json\nimport re\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\nSCHEMA_VERSION = 1\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef canonical_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()\n\n\ndef read_views(video, start_s, duration_s):\n    capture = cv2.VideoCapture(str(video))\n    try:\n        if not capture.isOpened():\n            raise ValueError(\'Cannot decode source video.\')\n        width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))\n        height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))\n        fps = float(capture.get(cv2.CAP_PROP_FPS))\n        frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))\n        if fps <= 0 or min(width, height, frames) <= 0:\n            raise ValueError(\'Invalid video metadata.\')\n        start, end = round(start_s * fps), round((start_s + duration_s) * fps)\n        if start < 0 or end <= start or end > frames:\n            raise ValueError(\'Requested interval is outside the source video.\')\n        views = []\n        indices = np.unique(np.linspace(start, end - 1, 5).astype(int))\n        for index in indices:\n            capture.set(cv2.CAP_PROP_POS_FRAMES, int(index))\n            ok, frame = capture.read()\n            if not ok:\n                raise ValueError(f\'Cannot decode view-check frame {index}.\')\n            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)\n            views.append(cv2.resize(gray, (960, round(height * 960 / width))))\n        return {\'width\': width, \'height\': height, \'fps\': fps, \'frames\': frames,\n                \'start_frame\': start, \'end_frame\': end,\n                \'view_check_frames\': indices.tolist()}, views\n    finally:\n        capture.release()\n\n\ndef compare_views(reference, current):\n    """Reject uncertain matches as well as sufficiently large shifts, zooms or rotations.\n\n    This is a sampled visual consistency check, not proof that the camera never moved.\n    Exclude timestamps, borders and foreground overlays; RANSAC discounts moving vehicles.\n    """\n    if reference is None or current is None or reference.shape != current.shape:\n        return {\'valid\': False, \'reason\': \'view_image_shape_changed\'}\n    height, width = reference.shape\n    mask = np.zeros_like(reference, dtype=np.uint8)\n    mask[round(.15 * height):round(.88 * height), round(.03 * width):round(.97 * width)] = 255\n    detector = cv2.ORB_create(nfeatures=5000, fastThreshold=12)\n    kp1, desc1 = detector.detectAndCompute(reference, mask)\n    kp2, desc2 = detector.detectAndCompute(current, mask)\n    if desc1 is None or desc2 is None:\n        return {\'valid\': False, \'reason\': \'insufficient_static_features\'}\n    pairs = cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(desc1, desc2, k=2)\n    matches = [a for pair in pairs if len(pair) == 2 for a, b in [pair] if a.distance < .7 * b.distance]\n    if len(matches) < 25:\n        return {\'valid\': False, \'reason\': \'insufficient_view_matches\', \'matches\': len(matches)}\n    a = np.float32([kp1[m.queryIdx].pt for m in matches])\n    b = np.float32([kp2[m.trainIdx].pt for m in matches])\n    matrix, inliers = cv2.findHomography(a, b, cv2.RANSAC, 3.0)\n    if matrix is None or inliers is None or not np.isfinite(matrix).all():\n        return {\'valid\': False, \'reason\': \'view_alignment_failed\'}\n    selected = inliers.ravel().astype(bool)\n    count = int(selected.sum())\n    coverage = min(abs(cv2.contourArea(cv2.convexHull(p[selected]))) / (width * height)\n                   for p in [a, b]) if count >= 3 else 0\n    corners = np.float32([[0, 0], [width, 0], [width, height], [0, height]])\n    projected = cv2.perspectiveTransform(corners[None], matrix)[0]\n    displacement = float(np.max(np.linalg.norm((projected - corners) / [width, height], axis=1)))\n    valid = count >= 25 and count / len(matches) >= .5 and coverage >= .10 and displacement <= .012\n    return {\'valid\': bool(valid), \'reason\': \'consistent_view\' if valid else \'view_changed_or_uncertain\',\n            \'matches\': len(matches), \'inliers\': count, \'feature_coverage_fraction\': coverage,\n            \'max_normalized_corner_displacement\': displacement}\n\n\ndef stable_view(views):\n    comparisons = [compare_views(views[0], view) for view in views[1:]]\n    return {\'valid\': bool(comparisons) and all(item[\'valid\'] for item in comparisons),\n            \'comparisons\': comparisons,\n            \'limitation\': \'Five sampled frames cannot exclude brief camera movement between samples.\'}\n\n\ndef valid_calibration(calibration):\n    if not isinstance(calibration, dict):\n        return False\n    accepted = 0\n    try:\n        for side in [\'left\', \'right\']:\n            mapping = calibration.get(side, {})\n            if not mapping.get(\'accepted\'):\n                continue\n            accepted += 1\n            matrices = [mapping[\'H_image_to_experimental_meters\'], *mapping[\'alternative_mappings\']]\n            if len(matrices) < 2:\n                return False\n            for value in matrices:\n                matrix = np.array(value, dtype=float)\n                if matrix.shape != (3, 3) or not np.isfinite(matrix).all() or abs(np.linalg.det(matrix)) < 1e-12:\n                    return False\n            hull = np.array(mapping[\'support_hull_image\'], dtype=np.float32)\n            if hull.ndim != 2 or hull.shape[1] != 2 or len(hull) < 3 or not np.isfinite(hull).all():\n                return False\n            if not cv2.isContourConvex(hull) or abs(cv2.contourArea(hull)) < 300:\n                return False\n    except (KeyError, TypeError, ValueError, cv2.error):\n        return False\n    return accepted > 0\n\n\nclass CalibrationCache:\n    def __init__(self, directory, camera_id):\n        if not camera_id.strip():\n            raise ValueError(\'A nonempty camera ID is required.\')\n        slug = re.sub(r\'[^a-zA-Z0-9_-]+\', \'-\', camera_id).strip(\'-\')[:48] or \'camera\'\n        self.camera_id = camera_id\n        self.directory = Path(directory) / (slug + \'-\' + canonical_hash(camera_id)[:12])\n        self.path = self.directory / \'calibration_cache.json\'\n\n    def load(self, signature, views):\n        if not self.path.exists():\n            return None, {\'status\': \'miss\', \'reason\': \'no_saved_calibration\'}\n        try:\n            entry = json.loads(self.path.read_text(encoding=\'utf-8\'))\n            if entry[\'schema_version\'] != SCHEMA_VERSION or entry[\'camera_id\'] != self.camera_id:\n                return None, {\'status\': \'miss\', \'reason\': \'cache_schema_or_camera_changed\'}\n            if entry[\'signature\'] != signature:\n                return None, {\'status\': \'miss\', \'reason\': \'resolution_model_or_calibration_settings_changed\'}\n            if canonical_hash(entry[\'calibration\']) != entry[\'calibration_sha256\'] or not valid_calibration(entry[\'calibration\']):\n                return None, {\'status\': \'miss\', \'reason\': \'invalid_calibration_data\'}\n            view_path = self.directory / \'reference_view.png\'\n            if file_sha256(view_path) != entry[\'reference_sha256\']:\n                return None, {\'status\': \'miss\', \'reason\': \'invalid_reference_image\'}\n            reference = cv2.imread(str(view_path), cv2.IMREAD_GRAYSCALE)\n            checks = [compare_views(reference, current) for current in views]\n            if not all(check[\'valid\'] for check in checks):\n                return None, {\'status\': \'miss\', \'reason\': \'camera_view_changed_or_uncertain\', \'view_checks\': checks}\n            return entry[\'calibration\'], {\'status\': \'hit\', \'reason\': \'saved_calibration_matches_camera_view\',\n                                          \'view_checks\': checks}\n        except (OSError, ValueError, KeyError, TypeError, cv2.error):\n            return None, {\'status\': \'miss\', \'reason\': \'cache_unreadable_or_corrupt\'}\n\n    def save(self, signature, reference, calibration, provenance):\n        if not valid_calibration(calibration):\n            raise ValueError(\'No usable road mapping; calibration will not be cached.\')\n        self.directory.mkdir(parents=True, exist_ok=True)\n        view_path = self.directory / \'reference_view.png\'\n        if not cv2.imwrite(str(view_path), reference):\n            raise OSError(\'Could not save camera reference.\')\n        entry = {\'schema_version\': SCHEMA_VERSION, \'camera_id\': self.camera_id,\n                 \'signature\': signature, \'calibration\': calibration,\n                 \'calibration_sha256\': canonical_hash(calibration),\n                 \'reference_sha256\': file_sha256(view_path), \'provenance\': provenance,\n                 \'accuracy_validated\': False}\n        temporary = self.path.with_suffix(\'.tmp\')\n        temporary.write_text(json.dumps(entry, indent=2, allow_nan=False), encoding=\'utf-8\')\n        temporary.replace(self.path)\n', 'calibrate_speed.py': '"""Experimental local road rectification from learned cuboids and explicit car-size priors.\n\nIndependent implementation of rectangle/aspect/rigidity fitting, inspired by\nSochor et al. (2017) and the UrbanOmniDetect (2026) ground-footprint approach.\nRanges are sensitivity envelopes, not confidence intervals or accuracy claims.\n"""\nfrom pathlib import Path\nfrom collections import defaultdict\nimport csv\nimport json\nimport math\nimport os\nimport cv2\nimport numpy as np\nfrom scipy.optimize import least_squares\n\nROOT = Path(os.environ.get(\'FYP_SPEED_ROOT\',str(Path(__file__).resolve().parent)))\ndownloaded_geometry = ROOT / \'outputs\' / \'geometry\'\ndefault_geometry = downloaded_geometry if downloaded_geometry.exists() else ROOT / \'output\' / \'inputs\' / \'geometry\'\nGEOMETRY = Path(os.environ.get(\'FYP_GEOMETRY_DIR\', str(default_geometry)))\nOUT = Path(os.environ.get(\'FYP_SPEED_OUTPUT\',str(ROOT / \'outputs\' / \'speed_experiment\')))\nLENGTH_M, WIDTH_M = 4.5, 1.8\nDIMENSION_SCENARIOS = [(4.5,1.8),(3.2,1.45),(3.2,2.0),(5.2,1.45),(5.2,2.0)]\n\n\ndef project(points, H):\n    p = np.asarray(points, dtype=float)\n    flat = p.reshape(-1,2)\n    result = np.c_[flat,np.ones(len(flat))] @ H.T\n    denominator = result[:,2]\n    safe = np.where(np.abs(denominator)<1e-8, np.where(denominator>=0,1e-8,-1e-8),denominator)\n    return (result[:,:2]/safe[:,None]).reshape(p.shape)\n\n\ndef compose(theta, normalizer):\n    p,q,a,b = theta\n    return np.array([[np.exp(a),b,0],[0,1,0],[p,q,1]]) @ normalizer\n\n\ndef measures(quads):\n    edges = np.roll(quads,-1,axis=1)-quads\n    sizes = np.linalg.norm(edges,axis=2)+1e-9\n    unit = edges/sizes[:,:,None]\n    cosines = np.sum(unit*np.roll(unit,-1,axis=1),axis=2)\n    opposite = unit[:,:2,0]*unit[:,2:,1]-unit[:,:2,1]*unit[:,2:,0]\n    lengths=(sizes[:,0]+sizes[:,2])/2\n    widths=(sizes[:,1]+sizes[:,3])/2\n    return cosines,opposite,lengths,widths\n\n\ndef fit_mapping(records, width, height, length_m=LENGTH_M, width_m=WIDTH_M, fast=False):\n    quads=np.array([r[\'quad\'] for r in records],dtype=float)\n    ids=np.array([r[\'track_id\'] for r in records])\n    unique=np.unique(ids)\n    normalizer=np.array([[2/width,0,-1],[0,2/width,-height/width],[0,0,1]])\n    weights=np.array([1/math.sqrt(np.sum(ids==i)) for i in ids])\n    weights/=np.mean(weights)\n    groups=[np.flatnonzero(ids==i) for i in unique]\n\n    def residual(theta):\n        H=compose(theta,normalizer)\n        world=project(quads,H)\n        angles,parallel,lengths,widths=measures(world)\n        rigidity=np.zeros(len(records))\n        for group in groups:\n            rigidity[group]=np.log(lengths[group]/np.median(lengths[group]))\n        shape=np.log((lengths/widths)/(length_m/width_m))\n        geometric=np.c_[angles,parallel*0.5,shape*0.65,rigidity*0.5]*weights[:,None]\n        den=(np.c_[quads.reshape(-1,2),np.ones(len(quads)*4)]@H.T)[:,2]\n        pole=np.maximum(0,0.08-den)*15\n        return np.r_[geometric.ravel(),pole]\n\n    starts=[(0,q,a,0) for q in ([0,1,2] if not fast else [1,2]) for a in ([0,0.7] if not fast else [0])]\n    fits=[]\n    for theta in starts:\n        result=least_squares(residual,theta,bounds=([-4,-4,-2.5,-4],[4,4,2.5,4]),\n                             loss=\'soft_l1\',f_scale=0.12,max_nfev=250)\n        fits.append((float(np.mean(residual(result.x)**2)),result))\n    _,result=min(fits,key=lambda item:item[0])\n    H=compose(result.x,normalizer)\n    world=project(quads,H)\n    _,_,lengths,_=measures(world)\n    # Give each track one vote; correlated video frames do not count as independent cars.\n    scale=length_m/np.median([np.median(lengths[g]) for g in groups])\n    H=np.diag([scale,scale,1])@H\n    world=project(quads,H)\n    directions=world[:,1]-world[:,0]\n    directions/=np.linalg.norm(directions,axis=1)[:,None]\n    directions[directions[:,1]<0]*=-1\n    direction=np.median(directions,axis=0)\n    angle=math.pi/2-math.atan2(direction[1],direction[0])\n    rotation=np.array([[math.cos(angle),-math.sin(angle),0],[math.sin(angle),math.cos(angle),0],[0,0,1]])\n    H=rotation@H\n    center=np.median(project(quads,H).mean(axis=1),axis=0)\n    H=np.array([[1,0,-center[0]],[0,1,-center[1]],[0,0,1]])@H\n    H/=np.linalg.norm(H[2,:2]) if np.linalg.norm(H[2,:2])>0 else H[2,2]\n    angle_error,_,lengths,widths=measures(project(quads,H))\n    singular=np.linalg.svd(result.jac,compute_uv=False)\n    diagnostics={\'fit_track_ids\':[int(i) for i in unique],\'fit_observations\':len(records),\n       \'median_angle_error_deg\':float(np.median(np.degrees(np.arcsin(np.clip(np.abs(angle_error),0,1))))),\n       \'median_length_m\':float(np.median(lengths)),\'median_width_m\':float(np.median(widths)),\n       \'parameter_condition_number\':float(singular[0]/max(singular[-1],1e-12)),\n       \'optimizer_success\':bool(result.success),\'theta\':result.x.tolist(),\n       \'dimension_prior_length_m\':length_m,\'dimension_prior_width_m\':width_m}\n    return H,diagnostics\n\n\ndef prepare(observations, indices, width, height):\n    by_id=defaultdict(list)\n    for r in observations:by_id[r[\'track_id\']].append(r)\n    records=[]\n    for tid,group in by_id.items():\n        group.sort(key=lambda r:r[\'time_s\'])\n        centers=np.array([[(r[\'baseline_box\'][0]+r[\'baseline_box\'][2])/2,\n                           (r[\'baseline_box\'][1]+r[\'baseline_box\'][3])/2] for r in group])\n        if len(group)<8 or np.linalg.norm(centers[-1]-centers[0])<80:continue\n        side=\'left\' if np.median(centers[:,0])<width/2 else \'right\'\n        for j,r in enumerate(group):\n            p=np.array(r[\'keypoints\'])[indices]\n            if p.shape[1]>2 and np.min(p[:,2])<0.30:continue\n            if r[\'geometry_confidence\']<0.25 or r[\'association_iou\']<0.40:continue\n            quad=p[:,:2]\n            if not np.isfinite(quad).all():continue\n            quad=quad[np.argsort(np.arctan2(*(quad-quad.mean(axis=0))[:,::-1].T))]\n            if not cv2.isContourConvex(quad.astype(np.float32)):continue\n            if abs(cv2.contourArea(quad.astype(np.float32)))<300:continue\n            if np.min(np.linalg.norm(np.roll(quad,-1,axis=0)-quad,axis=1))<8:continue\n            cx,cy=quad.mean(axis=0)\n            if not (0.20*height<cy<0.82*height and 0.02*width<cx<0.98*width):continue\n            if np.min(quad[:,0])<4 or np.max(quad[:,0])>width-4 or np.max(quad[:,1])>height-4:continue\n            lo=max(0,j-3);hi=min(len(group)-1,j+3)\n            motion=centers[hi]-centers[lo]\n            if np.linalg.norm(motion)<5:continue\n            edge=np.roll(quad,-1,axis=0)-quad\n            alignment=np.abs(edge@motion)/(np.linalg.norm(edge,axis=1)*np.linalg.norm(motion)+1e-9)\n            scores=[np.mean(alignment[[0,2]]),np.mean(alignment[[1,3]])]\n            if abs(scores[0]-scores[1])<0.08:continue\n            if scores[1]>scores[0]:quad=np.roll(quad,-1,axis=0)\n            records.append({\'track_id\':int(tid),\'time_s\':r[\'time_s\'],\'frame_index\':r[\'frame_index\'],\n                            \'side\':side,\'quad\':quad.tolist()})\n    return records\n\n\ndef estimate_track(records,H):\n    times=np.array([r[\'time_s\'] for r in records],dtype=float)\n    quads=np.array([r[\'quad\'] for r in records],dtype=float)\n    world=project(quads,H)\n    centers=world.mean(axis=1)\n    t=times-times.mean()\n    initial=np.r_[np.polyfit(t,centers[:,0],1),np.polyfit(t,centers[:,1],1)]\n    def residual(p):return np.r_[(p[0]*t+p[1]-centers[:,0]),(p[2]*t+p[3]-centers[:,1])]\n    fitted=least_squares(residual,initial,loss=\'soft_l1\',f_scale=0.4)\n    speed=float(np.hypot(fitted.x[0],fitted.x[2])*3.6)\n    rms=float(np.sqrt(np.mean(residual(fitted.x)**2)))\n    angles,_,lengths,widths=measures(world)\n    length_cv=float(1.4826*np.median(np.abs(lengths-np.median(lengths)))/np.median(lengths))\n    angle_error=float(np.median(np.degrees(np.arcsin(np.clip(np.abs(angles),0,1)))))\n    midpoint=(times[0]+times[-1])/2\n    half_speeds=[]\n    for mask in [times<=midpoint,times>midpoint]:\n        if mask.sum()>=4:\n            velocity=np.array([np.polyfit(times[mask]-times[mask].mean(),centers[mask,k],1)[0] for k in [0,1]])\n            half_speeds.append(float(np.linalg.norm(velocity)*3.6))\n    drift=abs(half_speeds[0]-half_speeds[1])/max(speed,1) if len(half_speeds)==2 else None\n    return {\'experimental_kmh\':speed,\'trajectory_fit_rms_m\':rms,\'length_relative_mad\':length_cv,\n            \'angle_error_deg\':angle_error,\'half_span_speed_relative_difference\':drift,\n            \'median_inferred_length_m\':float(np.median(lengths)),\'median_inferred_width_m\':float(np.median(widths))}\n\n\ndef sample_balanced(records, maximum=10):\n    groups=defaultdict(list)\n    for r in records:groups[r[\'track_id\']].append(r)\n    return [group[i] for group in groups.values() for i in np.unique(np.linspace(0,len(group)-1,min(len(group),maximum)).astype(int))]\n\n\ndef fit_road_mappings(records, report):\n    width,height=report[\'width\'],report[\'height\']\n    calibration={}\n    for side in [\'left\',\'right\']:\n        local=[r for r in records if r[\'side\']==side]\n        ids=sorted(set(r[\'track_id\'] for r in local))\n        if len(ids)<3:\n            calibration[side]={\'accepted\':False,\'reason\':\'Fewer than three qualifying car tracks\',\'track_ids\':ids}\n            continue\n        # Deterministic holdout by track identity; held-out predicted boxes are consistency evidence only.\n        held_ids=ids[::3]\n        fit_ids=[i for i in ids if i not in held_ids]\n        if len(fit_ids)<3:fit_ids=ids;held_ids=[]\n        fitting=sample_balanced([r for r in local if r[\'track_id\'] in fit_ids])\n        H,diagnostics=fit_mapping(fitting,width,height)\n        alternatives=[]\n        for length_m,width_m in DIMENSION_SCENARIOS[1:]:\n            alternative,_=fit_mapping(fitting,width,height,length_m,width_m,fast=True)\n            alternatives.append(alternative)\n        rng=np.random.default_rng(42)\n        for _ in range(10):\n            selected=set(rng.choice(fit_ids,size=max(3,round(len(fit_ids)*0.75)),replace=False).tolist())\n            subset=[r for r in fitting if r[\'track_id\'] in selected]\n            alternative,_=fit_mapping(subset,width,height,fast=True)\n            alternatives.append(alternative)\n        held=[r for r in local if r[\'track_id\'] in held_ids]\n        held_angle=None\n        if held:\n            angles,_,_,_=measures(project(np.array([r[\'quad\'] for r in held]),H))\n            held_angle=float(np.median(np.degrees(np.arcsin(np.clip(np.abs(angles),0,1)))))\n        accepted=diagnostics[\'median_angle_error_deg\']<20 and (held_angle is None or held_angle<25)\n        accepted=accepted and report[\'ground_order_vote_agreement\']>=0.8 and diagnostics[\'parameter_condition_number\']<1e5\n        diagnostics.update({\'accepted\':bool(accepted),\'held_out_track_ids\':held_ids,\n          \'held_out_median_angle_error_deg\':held_angle,\'H_image_to_experimental_meters\':H.tolist(),\n          \'alternative_mappings\':[a.tolist() for a in alternatives],\n          \'interpretation\':\'Approximate local plane and dimension-prior mapping; no measured accuracy.\'})\n        calibration[side]=diagnostics\n        diagnostics[\'support_hull_image\'] = cv2.convexHull(\n            np.array([r[\'quad\'] for r in local], dtype=np.float32).reshape(-1, 2)).reshape(-1, 2).tolist()\n    return calibration\n\n\ndef calculate_speed_rows(records, calibration):\n    speed_rows=[]\n    for side in [\'left\',\'right\']:\n        diagnostics=calibration.get(side,{})\n        if not diagnostics.get(\'accepted\'):\n            continue\n        H=np.array(diagnostics[\'H_image_to_experimental_meters\'])\n        alternatives=[np.array(a) for a in diagnostics[\'alternative_mappings\']]\n        hull=np.array(diagnostics[\'support_hull_image\'], dtype=np.float32)\n        # Restrict reuse to the road region supported by the calibration footprints.\n        local=[r for r in records if r[\'side\']==side and all(\n            cv2.pointPolygonTest(hull, (float(p[0]),float(p[1])), True)>=-0.01 for p in r[\'quad\'])]\n        ids=sorted(set(r[\'track_id\'] for r in local))\n        for tid in ids:\n            group=sorted([r for r in local if r[\'track_id\']==tid],key=lambda r:r[\'time_s\'])\n            if len(group)<8 or group[-1][\'time_s\']-group[0][\'time_s\']<1.2:continue\n            nominal=estimate_track(group,H)\n            values=[nominal[\'experimental_kmh\']]+[estimate_track(group,a)[\'experimental_kmh\'] for a in alternatives]\n            low,high=float(min(values)),float(max(values))\n            sensitivity=(high-low)/max(nominal[\'experimental_kmh\'],1)\n            reasons=[]\n            if nominal[\'length_relative_mad\']>0.25:reasons.append(\'predicted_size_inconsistent\')\n            if nominal[\'trajectory_fit_rms_m\']>0.8:reasons.append(\'trajectory_not_consistent_with_local_plane\')\n            if nominal[\'angle_error_deg\']>25:reasons.append(\'footprint_not_rectangular_after_mapping\')\n            if nominal[\'half_span_speed_relative_difference\'] is not None and nominal[\'half_span_speed_relative_difference\']>0.35:reasons.append(\'speed_changes_or_geometry_drift_across_span\')\n            if sensitivity>0.8:reasons.append(\'high_calibration_sensitivity\')\n            speed_rows.append({\'track_id\':tid,\'side\':side,\'time_start_s\':group[0][\'time_s\'],\n              \'time_end_s\':group[-1][\'time_s\'],\'geometry_observations\':len(group),\n              \'quality_passed\':not bool(reasons),\'quality_reasons\':\';\'.join(reasons),\n              \'dimension_and_fit_sensitivity_low_kmh\':low,\'dimension_and_fit_sensitivity_high_kmh\':high,\n              **nominal})\n    return speed_rows\n\n\ndef run(calibration=None, geometry_dir=None, output_dir=None, calibration_reused=None):\n    geometry_dir=Path(geometry_dir) if geometry_dir is not None else GEOMETRY\n    output_dir=Path(output_dir) if output_dir is not None else OUT\n    output_dir.mkdir(parents=True,exist_ok=True)\n    report=json.loads((geometry_dir/\'geometry_run_report.json\').read_text())\n    observations=json.loads((geometry_dir/\'observations.json\').read_text())\n    records=prepare(observations,report[\'ground_indices\'],report[\'width\'],report[\'height\'])\n    reused=(calibration is not None) if calibration_reused is None else calibration_reused\n    if calibration is None:\n        calibration=fit_road_mappings(records, report)\n    before_support_filter=len(records)\n    records=[r for r in records if calibration.get(r[\'side\'],{}).get(\'accepted\') and all(\n        cv2.pointPolygonTest(np.array(calibration[r[\'side\']][\'support_hull_image\'],dtype=np.float32),\n                             (float(p[0]),float(p[1])),True)>=-0.01 for p in r[\'quad\'])]\n    speed_rows=calculate_speed_rows(records, calibration)\n    (output_dir/\'filtered_ground_footprints.json\').write_text(json.dumps(records),encoding=\'utf-8\')\n    (output_dir/\'calibration.json\').write_text(json.dumps(calibration,indent=2),encoding=\'utf-8\')\n    fields=[\'track_id\',\'side\',\'time_start_s\',\'time_end_s\',\'geometry_observations\',\n            \'quality_passed\',\'quality_reasons\',\'dimension_and_fit_sensitivity_low_kmh\',\n            \'dimension_and_fit_sensitivity_high_kmh\',\'experimental_kmh\',\'trajectory_fit_rms_m\',\n            \'length_relative_mad\',\'angle_error_deg\',\'half_span_speed_relative_difference\',\n            \'median_inferred_length_m\',\'median_inferred_width_m\']\n    for name, rows in [(\'experimental_speeds.csv\',speed_rows),\n                       (\'qualified_experimental_speeds.csv\',[r for r in speed_rows if r[\'quality_passed\']])]:\n        with (output_dir/name).open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n            writer=csv.DictWriter(f,fieldnames=fields)\n            writer.writeheader()\n            writer.writerows(rows)\n    api_results=[{\'track_id\':r[\'track_id\'],\'carriageway\':r[\'side\'],\n      \'time_interval_s\':[r[\'time_start_s\'],r[\'time_end_s\']],\n      \'experimental_kmh\':r[\'experimental_kmh\'],\n      \'sensitivity_range_kmh\':[r[\'dimension_and_fit_sensitivity_low_kmh\'],r[\'dimension_and_fit_sensitivity_high_kmh\']],\n      \'range_type\':\'dimension_and_calibration_sensitivity\',\'accuracy_validated\':False,\n      \'method\':\'learned_3d_ground_footprints_with_assumed_vehicle_dimensions\'}\n      for r in speed_rows if r[\'quality_passed\']]\n    (output_dir/\'vehicle_speed_estimates.json\').write_text(json.dumps(api_results,indent=2),encoding=\'utf-8\')\n    result={\'calibration_reused\':reused,\'source_geometry_observations\':len(observations),\'filtered_ground_footprints\':len(records),\n      \'footprints_outside_usable_calibration\':before_support_filter-len(records),\n      \'geometry_model_filename\':report[\'model_filename\'],\'geometry_model_sha256\':report[\'model_sha256\'],\n      \'ground_corner_indices\':report[\'ground_indices\'],\n      \'experimental_track_estimates\':len(speed_rows),\'quality_passed_track_estimates\':sum(r[\'quality_passed\'] for r in speed_rows),\n      \'nominal_car_length_m\':LENGTH_M,\'nominal_car_width_m\':WIDTH_M,\'dimension_scenarios_m\':DIMENSION_SCENARIOS,\n      \'range_type\':\'Dimension-prior and track-subset sensitivity envelope. Not a statistical confidence interval.\',\n      \'scope\':\'Local visible road regions on each carriageway; passenger-car hypotheses only.\',\n      \'accuracy_validated\':False,\'speed_units\':\'km/h under assumed vehicle dimensions\',\n      \'limitations\':[\'No independent speed or road-distance ground truth.\',\n        \'Ground corners are model predictions; hidden corners can be wrong.\',\n        \'A local planar road and approximately constant speed over each fitted span are assumed.\',\n        \'Dimension priors are explicit analyst assumptions, not recognized make/model dimensions.\',\n        \'Sensitivity range does not bound unmodelled errors such as lens distortion or road grade.\']}\n    (output_dir/\'speed_run_report.json\').write_text(json.dumps(result,indent=2),encoding=\'utf-8\')\n    print(json.dumps(result,indent=2))\n    for r in speed_rows:print(r[\'track_id\'],r[\'side\'],round(r[\'experimental_kmh\'],1),r[\'quality_passed\'],r[\'quality_reasons\'])\n    return calibration,speed_rows,records\n\n\nif __name__==\'__main__\':run()\n', 'render_speed.py': '"""Render the offline experimental speed estimates and local mapping diagnostics."""\nfrom pathlib import Path\nimport csv\nimport json\nimport subprocess\nimport os\nfrom collections import defaultdict\nimport cv2\nimport numpy as np\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom calibrate_speed import ROOT, OUT, GEOMETRY, project\n\n\ndef draw_grid(frame, calibration, footprints):\n    height,width=frame.shape[:2]\n    for side,cal in calibration.items():\n        if not cal.get(\'accepted\'):continue\n        rows=[r for r in footprints if r[\'side\']==side]\n        if not rows:continue\n        H=np.array(cal[\'H_image_to_experimental_meters\'])\n        image_points=np.array([r[\'quad\'] for r in rows]).reshape(-1,2)\n        world=project(image_points,H)\n        low,high=np.percentile(world,[3,97],axis=0)\n        mask=np.zeros((height,width),np.uint8)\n        hull=cv2.convexHull(image_points.astype(np.float32)).astype(int)\n        cv2.fillConvexPoly(mask,hull,255)\n        overlay=np.zeros_like(frame)\n        inv=np.linalg.inv(H)\n        for axis in [0,1]:\n            for value in np.arange(np.floor(low[axis]/5)*5,high[axis]+5,5):\n                segment=np.array([[low[0]-5,low[1]-5],[high[0]+5,high[1]+5]])\n                segment[:,axis]=value\n                points=project(segment,inv)\n                if np.isfinite(points).all() and np.max(np.abs(points))<100000:\n                    a,b=points.astype(int);cv2.line(overlay,tuple(a),tuple(b),(60,200,255),2)\n        selected=(mask>0)&np.any(overlay>0,axis=2)\n        frame[selected]=overlay[selected]\n        cv2.polylines(frame,[hull],True,(255,200,60),2)\n    return frame\n\n\ndef main():\n    calibration=json.loads((OUT/\'calibration.json\').read_text())\n    footprints=json.loads((OUT/\'filtered_ground_footprints.json\').read_text())\n    csv_path=OUT/\'experimental_speeds.csv\'\n    speeds=list(csv.DictReader(csv_path.open(encoding=\'utf-8\'))) if csv_path.exists() else []\n    accepted={int(r[\'track_id\']):r for r in speeds if r[\'quality_passed\']==\'True\'}\n    baseline=defaultdict(list)\n    downloaded_tracks = ROOT / \'outputs\' / \'baseline\' / \'tracks.csv\'\n    default_tracks = downloaded_tracks if downloaded_tracks.exists() else ROOT / \'output\' / \'inputs\' / \'tracking\' / \'tracks.csv\'\n    with Path(os.environ.get(\'FYP_BASELINE_TRACKS\', str(default_tracks))).open(encoding=\'utf-8\') as f:\n        for row in csv.DictReader(f):baseline[int(row[\'frame_index\'])].append(row)\n    source_value = os.environ.get(\'FYP_SPEED_SOURCE\')\n    if not source_value:\n        raise SystemExit(\'Set FYP_SPEED_SOURCE to the original CCTV video path. See README.md.\')\n    source = Path(source_value)\n    capture=cv2.VideoCapture(str(source))\n    if not capture.isOpened():\n        raise SystemExit(f\'Cannot open source video: {source}\')\n    fps=capture.get(cv2.CAP_PROP_FPS)\n    width=int(capture.get(cv2.CAP_PROP_FRAME_WIDTH));height=int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))\n    start_s=float(os.environ.get(\'FYP_START_S\',150))\n    duration_s=float(os.environ.get(\'FYP_DURATION_S\',60))\n    capture.set(cv2.CAP_PROP_POS_FRAMES,round((start_s+duration_s*0.45)*fps));ok,reference=capture.read();assert ok\n    draw_grid(reference,calibration,footprints)\n    cv2.putText(reference,\'EXPERIMENTAL local mapping | 5 m grid under assumed car dimensions\',\n                (35,55),cv2.FONT_HERSHEY_SIMPLEX,1.05,(0,255,255),2)\n    cv2.imwrite(str(OUT/\'road_mapping.jpg\'),reference)\n    start,end=round(start_s*fps),round((start_s+duration_s)*fps)\n    capture.set(cv2.CAP_PROP_POS_FRAMES,start)\n    raw=OUT/\'experimental_speed_preview.avi\'\n    writer=cv2.VideoWriter(str(raw),cv2.VideoWriter_fourcc(*\'MJPG\'),fps,(1280,720));assert writer.isOpened()\n    labelled_frames=0\n    example_saved=False\n    try:\n        for index in range(start,end):\n            ok,frame=capture.read();assert ok,index\n            frame=cv2.resize(frame,(1280,720));sx,sy=1280/width,720/height\n            cv2.rectangle(frame,(0,0),(1280,66),(20,20,20),-1)\n            cv2.putText(frame,\'EXPERIMENTAL SPEED | approximate vehicle dimensions | accuracy unvalidated\',\n                        (15,25),cv2.FONT_HERSHEY_SIMPLEX,.62,(0,220,255),2)\n            cv2.putText(frame,\'Ranges show sensitivity to car size and calibration tracks; they are not confidence intervals.\',\n                        (15,51),cv2.FONT_HERSHEY_SIMPLEX,.52,(220,220,220),1)\n            has_label=False\n            for row in baseline.get(index,[]):\n                if not row[\'track_id\']:continue\n                tid=int(float(row[\'track_id\']));time_s=float(row[\'video_time_s\'])\n                speed=accepted.get(tid)\n                active=speed is not None and float(speed[\'time_start_s\'])<=time_s<=float(speed[\'time_end_s\'])\n                x1,y1,x2,y2=[int(float(row[k])*(sx if k.startswith(\'x\') else sy)) for k in [\'x1\',\'y1\',\'x2\',\'y2\']]\n                if active:\n                    label=(f\'ID {tid}: EXP {float(speed["experimental_kmh"]):.0f} \'\n                           f\'[{float(speed["dimension_and_fit_sensitivity_low_kmh"]):.0f}-\'\n                           f\'{float(speed["dimension_and_fit_sensitivity_high_kmh"]):.0f}] km/h\')\n                    color=(0,220,255);has_label=True\n                else:label=\'\';color=(170,170,170)\n                cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)\n                if not label:continue\n                tx=max(5,min(x1,1280-cv2.getTextSize(label,cv2.FONT_HERSHEY_SIMPLEX,.55,1)[0][0]-5))\n                ty=max(83,y1-8)\n                cv2.putText(frame,label,(tx,ty),cv2.FONT_HERSHEY_SIMPLEX,.55,(0,0,0),3)\n                cv2.putText(frame,label,(tx,ty),cv2.FONT_HERSHEY_SIMPLEX,.55,color,1)\n            cv2.putText(frame,f\'t={index/fps:.2f}s | Gray boxes: speed unavailable | Speeds average the qualifying span\',\n                        (15,701),cv2.FONT_HERSHEY_SIMPLEX,.55,(0,220,255),1)\n            if has_label:labelled_frames+=1\n            if has_label and not example_saved:\n                cv2.imwrite(str(OUT/\'speed_example.jpg\'),frame);example_saved=True\n            writer.write(frame)\n    finally:capture.release();writer.release()\n    preview=OUT/\'experimental_speed_preview.mp4\'\n    subprocess.run([\'ffmpeg\',\'-y\',\'-loglevel\',\'error\',\'-i\',str(raw),\'-c:v\',\'libx264\',\'-crf\',\'24\',\n                    \'-pix_fmt\',\'yuv420p\',\'-movflags\',\'+faststart\',str(preview)],check=True)\n    raw.unlink()\n\n    fig,axes=plt.subplots(1,2,figsize=(14,7))\n    for axis,side in zip(axes,[\'left\',\'right\']):\n        cal=calibration.get(side,{})\n        if cal.get(\'accepted\'):\n            H=np.array(cal[\'H_image_to_experimental_meters\'])\n            for tid in sorted(set(r[\'track_id\'] for r in footprints if r[\'side\']==side)):\n                group=sorted([r for r in footprints if r[\'side\']==side and r[\'track_id\']==tid],key=lambda r:r[\'time_s\'])\n                centers=project(np.array([r[\'quad\'] for r in group]),H).mean(axis=1)\n                axis.plot(centers[:,0],centers[:,1],\'.-\',label=f\'ID {tid}\')\n            axis.legend(fontsize=8);axis.set_aspect(\'equal\',adjustable=\'datalim\')\n        else:axis.text(.5,.5,\'Insufficient consistent geometry\',ha=\'center\',va=\'center\',transform=axis.transAxes)\n        axis.set_title(side.capitalize()+\' carriageway: approximate local plane\')\n        axis.set_xlabel(\'Assumption-based metres\');axis.set_ylabel(\'Assumption-based metres\');axis.grid(alpha=.2)\n    fig.suptitle(\'Experimental road mapping — arbitrary local origins; no independent distance validation\')\n    fig.tight_layout();fig.savefig(OUT/\'birdseye_trajectories.jpg\',dpi=150);plt.close(fig)\n    if accepted:\n        rows=list(accepted.values());nominal=np.array([float(r[\'experimental_kmh\']) for r in rows])\n        low=np.array([float(r[\'dimension_and_fit_sensitivity_low_kmh\']) for r in rows])\n        high=np.array([float(r[\'dimension_and_fit_sensitivity_high_kmh\']) for r in rows])\n        fig,axis=plt.subplots(figsize=(10,6))\n        axis.errorbar(nominal,np.arange(len(rows)),xerr=np.stack([nominal-low,high-nominal]),fmt=\'o\',capsize=5)\n        axis.set_yticks(np.arange(len(rows)),[f\'Track {r["track_id"]} ({r["side"]})\' for r in rows])\n        axis.set_xlabel(\'Experimental km/h under assumed dimensions\')\n        axis.set_title(\'Dimension and calibration-subset sensitivity envelopes\\nNot confidence intervals; speed accuracy remains unvalidated\')\n        axis.grid(axis=\'x\',alpha=.2);fig.tight_layout();fig.savefig(OUT/\'speed_sensitivity.jpg\',dpi=150);plt.close(fig)\n    rendering={\'preview_file\':preview.name,\'frames\':end-start,\'fps\':fps,\'duration_s\':(end-start)/fps,\n               \'frames_with_experimental_speed_labels\':labelled_frames,\'accepted_track_ids\':list(accepted)}\n    (OUT/\'render_report.json\').write_text(json.dumps(rendering,indent=2),encoding=\'utf-8\')\n    print(json.dumps(rendering,indent=2))\n\n\nif __name__==\'__main__\':main()\n', 'notebook/baseline.py': "# %% markdown\n# FYP CCTV vehicle detection and tracking baseline\n# Private source footage. First experiment: seconds 150–210, processed at full source FPS.\n# YOLO11s detects COCO vehicles; ByteTrack associates detections over time.\n# Outputs assess feasibility only. There are no labelled boxes, reference speeds, or metric calibration,\n# so detection AP, tracking accuracy and km/h cannot be claimed from this run.\n\n# %%\nimport subprocess\nimport sys\nimport os\nif os.environ.get('FYP_SKIP_INSTALL') != '1':\n    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',\n                           'ultralytics==8.4.175', 'lap>=0.5.12'])\n\n# %%\nfrom pathlib import Path\nfrom collections import Counter\nimport csv\nimport importlib.metadata\nimport json\nimport time\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport torch\nfrom ultralytics import YOLO\nfrom IPython.display import display, Video\n\nOUT = Path(os.environ.get('FYP_TRACKING_OUTPUT', '/kaggle/working/baseline'))\nOUT.mkdir(parents=True, exist_ok=True)\nVIDEO_NAME = 'A3_ch14_20260822105146_20260822110347.mp4'\nmatches = [Path(os.environ['FYP_SPEED_SOURCE'])] if os.environ.get('FYP_SPEED_SOURCE') else list(Path('/kaggle/input').rglob(VIDEO_NAME))\nassert len(matches) == 1, f'Expected exactly one source video; found {len(matches)}'\nSOURCE = matches[0]\nVIDEO_NAME = SOURCE.name\nSTART_S = float(os.environ.get('FYP_START_S', 150))\nDURATION_S = float(os.environ.get('FYP_DURATION_S', 60))\nDEVICE = os.environ.get('FYP_DEVICE') or (0 if torch.cuda.is_available() else 'cpu')\nif str(DEVICE).isdigit(): DEVICE = int(DEVICE)\nprint('Source:', SOURCE)\nprint('GPU:', torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU fallback')\nprint('Experiment interval:', START_S, 'to', START_S + DURATION_S, 'seconds')\n\ncapture = cv2.VideoCapture(str(SOURCE))\nassert capture.isOpened(), 'Cannot open HEVC source footage'\nfps = float(capture.get(cv2.CAP_PROP_FPS))\nwidth = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))\nheight = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))\ntotal_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))\nassert fps > 0 and width > 0 and height > 0\nstart_frame = round(START_S * fps)\nend_frame = min(total_frames, round((START_S + DURATION_S) * fps))\ncapture.set(cv2.CAP_PROP_POS_FRAMES, start_frame)\nassert abs(capture.get(cv2.CAP_PROP_POS_FRAMES) - start_frame) <= 1, 'Frame seek failed'\nprint({'width': width, 'height': height, 'fps': fps, 'frames': total_frames})\n\n# %% markdown\n# Bounding-box bottom centres are saved as provisional ground-contact proxies.\n# They can move with detection jitter and are not yet suitable for calibrated speeds.\n# Tracker IDs are associations, not verified identities or unique vehicle counts.\n\n# %%\ntracker_config = OUT / 'bytetrack_25fps.yaml'\ntracker_config.write_text('''tracker_type: bytetrack\ntrack_high_thresh: 0.25\ntrack_low_thresh: 0.10\nnew_track_thresh: 0.25\ntrack_buffer: 30\nmatch_thresh: 0.8\nfuse_score: True\n''', encoding='utf-8')\nmodel_dir = Path(os.environ.get('FYP_MODELS_DIR', '/kaggle/working'))\nmodel_dir.mkdir(parents=True, exist_ok=True)\nmodel = YOLO(str(model_dir / 'yolo11s.pt'))\n# ByteTrack's frame-rate-aware constructor uses 30 FPS by default in Ultralytics.\n# Rescale its buffer to retain approximately one second of missed tracks at 25 FPS.\ntracker_config.write_text(tracker_config.read_text().replace('track_buffer: 30',\n                           f'track_buffer: {round(fps)}'), encoding='utf-8')\npreview_size = (1280, round(height * 1280 / width))\nraw_preview = OUT / 'tracking_preview.avi'\nwriter = cv2.VideoWriter(str(raw_preview), cv2.VideoWriter_fourcc(*'MJPG'), fps, preview_size)\nassert writer.isOpened(), 'Preview writer failed'\nfields = ['frame_index', 'video_time_s', 'time_source', 'track_id', 'class_id',\n          'class_name', 'confidence', 'x1', 'y1', 'x2', 'y2', 'contact_x', 'contact_y']\nrows = []\nsample_images = []\ncandidate_crops = {}\nframe_count = 0\nprevious_pts = -1.0\nstart_wall = time.monotonic()\ntry:\n    for frame_index in range(start_frame, end_frame):\n        ok, frame = capture.read()\n        if not ok:\n            raise RuntimeError(f'Decode stopped unexpectedly at frame {frame_index}')\n        pts = float(capture.get(cv2.CAP_PROP_POS_MSEC)) / 1000.0\n        expected_time = frame_index / fps\n        use_pts = np.isfinite(pts) and pts > previous_pts and abs(pts - expected_time) < 1.0\n        timestamp = pts if use_pts else expected_time\n        time_source = 'decoder_pts' if use_pts else 'frame_index_over_fps'\n        previous_pts = pts\n        result = model.track(frame, persist=True, tracker=str(tracker_config),\n                             classes=[2, 3, 5, 7], conf=0.10, iou=0.7,\n                             imgsz=1280, device=DEVICE, verbose=False)[0]\n        boxes = result.boxes\n        coords = boxes.xyxy.cpu().numpy()\n        classes = boxes.cls.cpu().numpy().astype(int)\n        confidences = boxes.conf.cpu().numpy()\n        ids = boxes.id.cpu().numpy().astype(int) if boxes.id is not None else [None] * len(coords)\n        for box, cls, confidence, tid in zip(coords, classes, confidences, ids):\n            x1, y1, x2, y2 = [float(v) for v in box]\n            row = dict(zip(fields, [frame_index, timestamp, time_source,\n                       int(tid) if tid is not None else None, int(cls), model.names[int(cls)],\n                       float(confidence), x1, y1, x2, y2, (x1+x2)/2, y2]))\n            rows.append(row)\n            if tid is not None and cls == 2 and confidence >= 0.65:\n                area = max(0, x2-x1) * max(0, y2-y1)\n                if area > candidate_crops.get(int(tid), (0, None, None))[0] and area > 10000:\n                    crop = frame[max(0,int(y1)):min(height,int(y2)), max(0,int(x1)):min(width,int(x2))]\n                    if crop.size: candidate_crops[int(tid)] = (area, crop.copy(), row.copy())\n        annotated = result.plot()\n        cv2.putText(annotated, f't={timestamp:.2f}s | Detection/tracking only | No metric calibration',\n                    (25, height-30), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0,255,255), 2)\n        writer.write(cv2.resize(annotated, preview_size))\n        if frame_count % round(5*fps) == 0:\n            image_path = OUT / f'frame_{frame_index:06d}.jpg'\n            cv2.imwrite(str(image_path), annotated)\n            sample_images.append((timestamp, cv2.resize(annotated, preview_size)))\n        frame_count += 1\n        if frame_count % round(10*fps) == 0:\n            print(f'Processed {frame_count}/{end_frame-start_frame} frames; {len(rows)} observations', flush=True)\nfinally:\n    capture.release()\n    writer.release()\nelapsed_s = time.monotonic() - start_wall\nwith (OUT / 'tracks.csv').open('w', newline='', encoding='utf-8') as handle:\n    csv_writer = csv.DictWriter(handle, fieldnames=fields)\n    csv_writer.writeheader()\n    csv_writer.writerows(rows)\nassert frame_count == end_frame-start_frame, 'Incomplete processing'\nprint('Completed frames:', frame_count, 'Runtime seconds:', round(elapsed_s, 1))\n\n# %%\ndf = pd.DataFrame(rows, columns=fields)\ntracked = df.dropna(subset=['track_id']).copy()\ntrack_summaries = []\nfor tid, group in tracked.groupby('track_id', sort=True):\n    group = group.sort_values('frame_index')\n    first, last = group.iloc[0], group.iloc[-1]\n    duration = float(last.video_time_s-first.video_time_s)\n    displacement = float(np.hypot(last.contact_x-first.contact_x, last.contact_y-first.contact_y))\n    track_summaries.append({'track_id': int(tid), 'class_name': group.class_name.mode().iloc[0],\n       'observed_frames': len(group), 'first_video_time_s': float(first.video_time_s),\n       'last_video_time_s': float(last.video_time_s), 'span_s': duration,\n       'coverage_fraction': len(group)/(int(last.frame_index-first.frame_index)+1),\n       'mean_detection_confidence': float(group.confidence.mean()),\n       'endpoint_displacement_pixels': displacement,\n       'max_gap_frames': int(group.frame_index.diff().max()) if len(group)>1 else 0})\nsummary_fields = ['track_id','class_name','observed_frames','first_video_time_s',\n                  'last_video_time_s','span_s','coverage_fraction','mean_detection_confidence',\n                  'endpoint_displacement_pixels','max_gap_frames']\nsummary = pd.DataFrame(track_summaries, columns=summary_fields)\nsummary.to_csv(OUT/'track_summary.csv', index=False)\ndisplay(summary.sort_values('observed_frames', ascending=False).head(20))\n\nfig, axes = plt.subplots(3, 4, figsize=(20, 10))\nfor axis, sample in zip(axes.flat, sample_images):\n    timestamp, sample_image = sample\n    axis.imshow(cv2.cvtColor(sample_image, cv2.COLOR_BGR2RGB))\n    axis.set_title(f'Video t={timestamp:.1f}s')\nfor axis in axes.flat: axis.axis('off')\nfig.tight_layout()\nfig.savefig(OUT/'detection_contact_sheet.jpg', dpi=150)\nplt.show()\n\nif sample_images:\n    fig, axis = plt.subplots(figsize=(16, 9))\n    axis.imshow(cv2.cvtColor(sample_images[0][1], cv2.COLOR_BGR2RGB), extent=(0,width,height,0))\n    for tid, group in tracked.groupby('track_id'):\n        if len(group) >= 10:\n            axis.plot(group.contact_x, group.contact_y, linewidth=1.5)\n    axis.set_title('Provisional bounding-box contact trajectories — image coordinates only')\n    axis.set_xlim(0,width); axis.set_ylim(height,0)\n    fig.tight_layout(); fig.savefig(OUT/'trajectories.jpg', dpi=150)\n    plt.show()\n\ncrop_metadata = []\nfor tid, (area, crop, row) in sorted(candidate_crops.items(), key=lambda item: -item[1][0])[:12]:\n    crop_name = f'car_candidate_id_{tid}.jpg'\n    cv2.imwrite(str(OUT/crop_name), crop)\n    crop_metadata.append({'file': crop_name, 'box_area_pixels': area, **row})\n(OUT/'vehicle_crop_metadata.json').write_text(json.dumps(crop_metadata, indent=2), encoding='utf-8')\n\n# %%\npreview = OUT/'tracking_preview.mp4'\nconversion = subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(raw_preview),\n             '-c:v','libx264','-crf','24','-pix_fmt','yuv420p','-movflags','+faststart',str(preview)],\n             capture_output=True, text=True)\nif conversion.returncode == 0:\n    raw_preview.unlink()\nelse:\n    print('H.264 conversion failed; retained AVI:', conversion.stderr[-1000:])\n    preview = raw_preview\nreport = {'stage':'vehicle_detection_and_tracking_baseline', 'metric_speed_available':False,\n 'source_video':VIDEO_NAME, 'source_width':width, 'source_height':height, 'source_fps':fps,\n 'source_frames':total_frames, 'start_s':START_S, 'duration_s':DURATION_S,\n 'processed_frames':frame_count, 'detection_observations':len(df),\n 'tracked_observations':len(tracked), 'tracker_ids':len(summary),\n 'track_ids_with_at_least_25_observations':int((summary.observed_frames>=25).sum()),\n 'observations_by_class':dict(Counter(df.class_name)),\n 'timestamp_sources':dict(Counter(df.time_source)),\n 'inference_wall_seconds':elapsed_s, 'model':'yolo11s.pt', 'tracker':'ByteTrack',\n 'imgsz':1280, 'confidence_threshold':0.10, 'device':str(DEVICE),\n 'gpu':torch.cuda.get_device_name(0) if DEVICE==0 else None,\n 'versions':{p:importlib.metadata.version(p) for p in ['ultralytics','torch','lap','opencv-python']},\n 'preview_file':preview.name,\n 'limitations':['No ground-truth annotations: no AP, IDF1 or speed accuracy measured.',\n 'Tracker IDs are not verified unique vehicles.',\n 'Box bottom centres are provisional contact proxies.',\n 'No metric scale or road-plane calibration has been estimated.',\n 'Road grade and crest may require separate local planes or a 3D road model.']}\n(OUT/'run_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')\nprint(json.dumps(report, indent=2))\nif preview.suffix=='.mp4': display(Video(str(preview), embed=True, width=960))\n\n# %% markdown\n# Next: inspect tracking continuity and car crops, choose local road regions, then estimate\n# camera orientation and metric scale using vehicle geometry. The cited Sochor et al. method\n# matches detected boxes to rendered 3D vehicle models; this notebook does not implement that yet.\n# https://arxiv.org/abs/1702.06451\n", 'geometry_notebook/extract_geometry.py': "# %% markdown\n# FYP automatic vehicle geometry experiment\n# A pretrained UrbanOmniDetect (CVPR 2026 DriveX) model predicts 8 projected 3D cuboid corners.\n# Predictions are associated with the original baseline tracker IDs. No metric speeds are\n# assigned here: approximate dimension priors and geometry consistency are evaluated separately.\n# Source and citation: https://github.com/mkturkcan/urbanomnidetect\n# Turkcan, Gumaste and Kostic, Calibration-Free View-Agnostic Monocular 3D Object Detection\n# for Urban Scenes, CVPR Workshops 2026, pp. 786–795.\n\n# %%\nimport subprocess, sys, os\nif os.environ.get('FYP_SKIP_INSTALL') != '1':\n    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.175'])\n\n# %%\nfrom pathlib import Path\nimport json, time, urllib.request, hashlib\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom scipy.optimize import linear_sum_assignment\nfrom ultralytics import YOLO\nfrom IPython.display import display\nimport matplotlib.pyplot as plt\n\nOUT = Path(os.environ.get('FYP_GEOMETRY_OUTPUT', '/kaggle/working/geometry'))\nOUT.mkdir(parents=True, exist_ok=True)\nSOURCE = [Path(os.environ['FYP_SPEED_SOURCE'])] if os.environ.get('FYP_SPEED_SOURCE') else list(Path('/kaggle/input').rglob('A3_ch14_20260822105146_20260822110347.mp4'))\nTRACKS = [Path(os.environ['FYP_TRACKS_CSV'])] if os.environ.get('FYP_TRACKS_CSV') else list(Path('/kaggle/input').rglob('tracks.csv'))\nassert len(SOURCE)==1 and len(TRACKS)==1, (SOURCE,TRACKS)\nSOURCE, TRACKS = SOURCE[0], TRACKS[0]\nREPO='mehmetkeremturkcan/UrbanOmniDetect'\nREVISION='2701a2eec9a73afb5ffd852740d125dfcd065ce5'\nFILENAME='checkpoints/urbanomnidetect_yolo11x-p2_1920.pt'\nweights=Path(os.environ.get('FYP_MODELS_DIR', '/kaggle/working'))/'urban_vehicle_geometry.pt'\nweights.parent.mkdir(parents=True, exist_ok=True)\nEXPECTED_SHA='9aa55280a5ef1c9d89c0acb5baeb974b90fbdf3c1deead1413b24d7af249226d'\nif not weights.exists() or hashlib.sha256(weights.read_bytes()).hexdigest()!=EXPECTED_SHA:\n    urllib.request.urlretrieve(f'https://huggingface.co/{REPO}/resolve/{REVISION}/{FILENAME}',weights)\nweight_sha=hashlib.sha256(weights.read_bytes()).hexdigest()\nassert weight_sha==EXPECTED_SHA,'Unexpected geometry checkpoint checksum'\nmodel=YOLO(str(weights))\ncar_classes=[int(k) for k,v in model.names.items() if v.lower()=='car']\nassert car_classes,'Model has no car class'\ndevice=os.environ.get('FYP_DEVICE') or (0 if torch.cuda.is_available() else 'cpu')\nif str(device).isdigit(): device=int(device)\nprint('Model:',REPO,REVISION,'Names:',model.names,'Device:',device)\ndf=pd.read_csv(TRACKS).dropna(subset=['track_id'])\ndf['track_id']=df.track_id.astype(int)\neligible=[]\nfor tid,group in df.groupby('track_id'):\n    if len(group)>=25 and (group.class_name=='car').mean()>=0.7:\n        eligible.append(tid)\ndf=df[df.track_id.isin(eligible)]\ndf=df[(df.frame_index%5==0)&(df.confidence>=0.3)&((df.y2-df.y1)>=60)]\nby_frame={int(n):g for n,g in df.groupby('frame_index')}\ncap=cv2.VideoCapture(str(SOURCE));assert cap.isOpened()\nfps=cap.get(cv2.CAP_PROP_FPS);width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))\nstart_s=float(os.environ.get('FYP_START_S',150))\nduration_s=float(os.environ.get('FYP_DURATION_S',60))\nstart,end=round(start_s*fps),round((start_s+duration_s)*fps)\ncap.set(cv2.CAP_PROP_POS_FRAMES,start)\nrecords=[];samples=[];total_predictions=0;frames_processed=0\nstart_wall=time.monotonic()\n\ndef iou_matrix(a,b):\n    lo=np.maximum(a[:,None,:2],b[None,:,:2]);hi=np.minimum(a[:,None,2:],b[None,:,2:])\n    inter=np.prod(np.maximum(0,hi-lo),axis=2)\n    area_a=np.prod(a[:,2:]-a[:,:2],axis=1);area_b=np.prod(b[:,2:]-b[:,:2],axis=1)\n    return inter/(area_a[:,None]+area_b[None,:]-inter+1e-9)\n\ntry:\n    for frame_index in range(start,end):\n        ok,frame=cap.read()\n        assert ok,f'Decode stopped at {frame_index}'\n        if frame_index not in by_frame:continue\n        result=model.predict(frame,device=device,imgsz=1920,conf=0.15,classes=car_classes,verbose=False)[0]\n        frames_processed+=1\n        total_predictions+=len(result.boxes)\n        if result.keypoints is None or not len(result.boxes):continue\n        baseline=by_frame[frame_index]\n        boxes=result.boxes.xyxy.cpu().numpy();scores=result.boxes.conf.cpu().numpy()\n        points=result.keypoints.data.cpu().numpy()\n        assert points.shape[1]==8,points.shape\n        overlap=iou_matrix(baseline[['x1','y1','x2','y2']].to_numpy(),boxes)\n        ii,jj=linear_sum_assignment(1-overlap)\n        for i,j in zip(ii,jj):\n            if overlap[i,j]<0.35:continue\n            row=baseline.iloc[i]\n            records.append({'frame_index':frame_index,'time_s':float(row.video_time_s),\n                'track_id':int(row.track_id),'baseline_confidence':float(row.confidence),\n                'geometry_confidence':float(scores[j]),'association_iou':float(overlap[i,j]),\n                'baseline_box':[float(row[k]) for k in ['x1','y1','x2','y2']],\n                'geometry_box':boxes[j].tolist(),'geometry_class':'car','keypoints':points[j].tolist()})\n        if frame_index%125==0 and len(samples)<12:\n            # Keep raw images too: keypoint quality must be checked against the source.\n            cv2.imwrite(str(OUT/f'raw_{frame_index:06d}.jpg'),frame)\n            annotated=result.plot()\n            for points_one in points:\n                corners=points_one[:,:2].astype(int)\n                for a,b in [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]:\n                    cv2.line(annotated,tuple(corners[a]),tuple(corners[b]),(0,220,255),2)\n            cv2.imwrite(str(OUT/f'pose_{frame_index:06d}.jpg'),annotated)\n            samples.append((frame_index/fps,cv2.resize(annotated,(1280,720))))\n        if frames_processed%50==0:print('Inference frames:',frames_processed,'Matched observations:',len(records),flush=True)\nfinally:\n    cap.release()\n\n(OUT/'observations.json').write_text(json.dumps(records),encoding='utf-8')\n# Resolve checkpoint corner-order discrepancy by voting, then lock for the entire run.\nif records:\n    p=np.array([r['keypoints'] for r in records])\n    vote=(p[:,:4,1].mean(axis=1)>p[:,4:,1].mean(axis=1))\n    first_ground=bool(vote.mean()>=0.5)\n    ground_indices=list(range(4)) if first_ground else list(range(4,8))\n    agreement=float(max(vote.mean(),1-vote.mean()))\nelse:ground_indices=[];agreement=0\nreport={'model_repo':REPO,'model_revision':REVISION,'model_filename':FILENAME,\n    'source_video':SOURCE.name,'start_s':start_s,'duration_s':duration_s,\n    'model_sha256':weight_sha,'model_names':model.names,'ground_indices':ground_indices,\n    'ground_order_vote_agreement':agreement,'width':width,'height':height,'fps':fps,\n    'inference_frames':frames_processed,'all_geometry_predictions':total_predictions,\n    'matched_observations':len(records),'matched_track_ids':len(set(r['track_id'] for r in records)),\n    'runtime_s':time.monotonic()-start_wall,'metric_calibration_available':False,'imgsz':1920,\n    'gpu':torch.cuda.get_device_name(0) if device==0 else None,\n    'limitations':['Predicted ground corners are learned estimates, not measured contacts.',\n    'No scene-specific keypoint labels or speed ground truth supplied.',\n    'Approximate metric scale must be labelled as a dimension-prior hypothesis.']}\n(OUT/'geometry_run_report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')\nprint(json.dumps(report,indent=2))\nfig,axes=plt.subplots(3,4,figsize=(20,10))\nfor ax,(timestamp,image) in zip(axes.flat,samples):\n    ax.imshow(cv2.cvtColor(image,cv2.COLOR_BGR2RGB));ax.set_title(f't={timestamp:.1f}s')\nfor ax in axes.flat:ax.axis('off')\nfig.tight_layout();fig.savefig(OUT/'geometry_contact_sheet.jpg',dpi=150);plt.show()\n"}
from pathlib import Path
root = Path('/kaggle/working/pipeline')
for name, source in sources.items():
    path = root / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding='utf-8')
sys.path.insert(0, str(root))
import workflow
import json
from unittest.mock import patch
videos = list(Path('/kaggle/input').rglob('A3_ch14_20260822105146_20260822110347.mp4'))
assert len(videos) == 1, videos
arguments = ['--video', str(videos[0]), '--camera-id', 'A3-CH14',
             '--work-dir', '/kaggle/working/workflow_outputs', '--device', '0']


In [ ]:
cold_code, cold_dir = workflow.run_workflow(workflow.parser().parse_args(arguments))
assert cold_code == 0, (cold_dir / 'workflow_report.json').read_text()
cold = json.loads((cold_dir / 'workflow_report.json').read_text())
assert cold['calibration_lookup']['status'] == 'miss'
assert any(s['action'] == 'model_inference' for s in cold['stages'])
with patch.object(workflow.speed, 'fit_road_mappings', side_effect=AssertionError('Warm run refitted calibration')):
    warm_code, warm_dir = workflow.run_workflow(workflow.parser().parse_args(arguments + ['--no-render']))
assert warm_code == 0, (warm_dir / 'workflow_report.json').read_text()
warm = json.loads((warm_dir / 'workflow_report.json').read_text())
assert warm['calibration_lookup']['status'] == 'hit'
assert cold['qualified_track_ids'] == warm['qualified_track_ids']
assert (cold_dir / 'speed/vehicle_speed_estimates.json').read_text() == (warm_dir / 'speed/vehicle_speed_estimates.json').read_text()
verification = {'cold_run': cold_dir.name, 'warm_run': warm_dir.name,
    'cold_cache_calibrated': True, 'fresh_gpu_inference_executed': True,
    'warm_cache_reused_without_fitting': True, 'qualified_results_identical': True,
    'qualified_track_ids': warm['qualified_track_ids'], 'accuracy_validated': False}
Path('/kaggle/working/workflow_verification.json').write_text(json.dumps(verification, indent=2))
print(json.dumps(verification, indent=2))


In [ ]:
from IPython.display import display, Image, Video
import pandas as pd
display(pd.read_csv(cold_dir / 'speed/qualified_experimental_speeds.csv'))
for name in ['road_mapping.jpg', 'speed_sensitivity.jpg']:
    display(Image(filename=str(cold_dir / 'speed' / name), width=1100))
display(Video(str(cold_dir / 'speed/experimental_speed_preview.mp4'), embed=True, width=1100))
